In [51]:
%%writefile task2_chunks.py
import re
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size-overlap):
        j = min(i+size, len(spans))
        result.append((offset+spans[i][0], offset+spans[j-1][1]))
        if j == len(spans):
            break
    return result
def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    cuts = [0] + [m.end() for m in
                  re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
          continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count+n <= size:
            pending, count = (pending[0], b), count+n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
            out.append(pending)
    return out
def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = (sentence_chunks(text) if strategy == "sentence"
                 else windows(text, 40, int(strategy)))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id,
                               start=a, end=b, text=text[a:b]))
    return result
if __name__ == "__main__":
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap was accepted")
    long = " ".join(["word"]*95) + "."
    assert all(len(long[a:b].split()) <= 40
               for a, b in sentence_chunks(long))
    docs = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ["0", "10", "sentence"]:
        for c in make_chunks(docs, strategy):
            assert c["text"] == docs[c["doc"]][c["start"]:c["end"]]
            print(strategy, c)


Writing task2_chunks.py


In [52]:
docs = {
    "attendance": """
    Students are expected to attend all scheduled classes.
    A minimum attendance of 75 percent is required to appear in final examinations.
    Students who arrive late may be marked absent by the instructor.
    Attendance records are maintained by the department.
    Students should check their attendance regularly.
    If a student has a genuine medical emergency, the department may review the case.
    Medical documents must be submitted for verification.
    The department will make the final decision after reviewing the evidence.
    Students should contact their class coordinator for attendance-related questions.
    Students can check their attendance through the university portal.
    Any mistake in the attendance record should be reported immediately.
    The instructor may correct errors after checking the class record.
    Students should inform their instructor if they cannot attend a class.
    Attendance rules apply to all regular classes and laboratory sessions.
    A student should maintain good attendance throughout the semester.
    The department may provide guidance to students with low attendance.
    """,

    "assignments": """
    Students must submit assignments before the announced deadline.
    Late submissions normally receive a penalty.
    However, students with verified medical emergencies may request an extension.
    The request must include valid medical evidence.
    The instructor reviews each request before approving an extension.
    Students should upload their assignments through the learning portal.
    The portal provides a submission confirmation.
    Students should keep a copy of that confirmation.
    Assignments submitted through unofficial channels may not be accepted.
    Students should read all assignment instructions carefully.
    Every assignment must include the student's name and roll number.
    Work should be completed independently unless group work is allowed.
    Students should check their files before uploading them.
    Incorrect or empty files may receive no marks.
    If the portal is unavailable, students should inform the instructor.
    The instructor may provide alternative submission instructions.
    Students should not copy another student's work.
    Plagiarism can result in academic penalties.
    Students should submit their work carefully.
    Students should submit their work carefully and follow all instructions.
    """,

    "library": """
    Students can borrow books using their valid university identification cards.
    Books must be returned on or before due date.
    Late returns may result in a fine.
    Students should handle library materials carefully.
    Damaged books must be reported to the library staff.
    Reference books are normally available for reading inside the library only.
    Students may request help from librarian.
    The library provides access to academic resources.
    Students should follow library rules during their visits.
    Students must maintain silence inside library.
    Food and drinks are not allowed near library materials.
    Borrowed books should not be shared with unauthorized users.
    Students can search for books using the library catalogue.
    The librarian can help students find relevant materials.
    Computers may be available for academic research.
    Students should return books to correct location.
    Lost books must be reported to library staff.
    Library facilities should be used responsibly by everyone.
    Students should submit their work carefully.
    """
}

for doc_id, text in docs.items():
    print(doc_id, ":", len(text.split()), "words")

attendance : 155 words
assignments : 161 words
library : 150 words


In [53]:
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)

    print("\nStrategy:", strategy)
    print("Total chunks:", len(chunks))

    for chunk in chunks:
        print(chunk["id"], "Words:", len(chunk["text"].split()))


Strategy: 0
Total chunks: 13
attendance:0 Words: 40
attendance:1 Words: 40
attendance:2 Words: 40
attendance:3 Words: 35
assignments:0 Words: 40
assignments:1 Words: 40
assignments:2 Words: 40
assignments:3 Words: 40
assignments:4 Words: 1
library:0 Words: 40
library:1 Words: 40
library:2 Words: 40
library:3 Words: 30

Strategy: 10
Total chunks: 16
attendance:0 Words: 40
attendance:1 Words: 40
attendance:2 Words: 40
attendance:3 Words: 40
attendance:4 Words: 35
assignments:0 Words: 40
assignments:1 Words: 40
assignments:2 Words: 40
assignments:3 Words: 40
assignments:4 Words: 40
assignments:5 Words: 11
library:0 Words: 40
library:1 Words: 40
library:2 Words: 40
library:3 Words: 40
library:4 Words: 30

Strategy: sentence
Total chunks: 14
attendance:0 Words: 39
attendance:1 Words: 38
attendance:2 Words: 38
attendance:3 Words: 40
assignments:0 Words: 40
assignments:1 Words: 39
assignments:2 Words: 36
assignments:3 Words: 36
assignments:4 Words: 10
library:0 Words: 32
library:1 Words: 33


In [54]:
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)

    print("\nStrategy:", strategy)

    for c in chunks:
        original = docs[c["doc"]][c["start"]:c["end"]]

        assert c["text"] == original

    print("All source offsets are correct!")


Strategy: 0
All source offsets are correct!

Strategy: 10
All source offsets are correct!

Strategy: sentence
All source offsets are correct!


In [55]:
for doc_id, text in docs.items():
    print(doc_id, ":", len(text.split()), "words")

attendance : 155 words
assignments : 161 words
library : 150 words


In [56]:
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)

    original_words = sum(len(text.split()) for text in docs.values())
    chunk_words = sum(len(c["text"].split()) for c in chunks)
    repeated_words = chunk_words - original_words

    print("\nStrategy:", strategy)
    print("Total chunks:", len(chunks))
    print("Original words:", original_words)
    print("Words in chunks:", chunk_words)
    print("Repeated words:", repeated_words)


Strategy: 0
Total chunks: 13
Original words: 466
Words in chunks: 466
Repeated words: 0

Strategy: 10
Total chunks: 16
Original words: 466
Words in chunks: 596
Repeated words: 130

Strategy: sentence
Total chunks: 14
Original words: 466
Words in chunks: 466
Repeated words: 0


In [57]:
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)

    print("\nStrategy:", strategy)

    for doc_id, text in docs.items():
        for sentence in text.split("."):
            sentence = sentence.strip()
            if "However" in sentence:
                found = any(
                    sentence in c["text"]
                    for c in chunks
                    if c["doc"] == doc_id
                )
                print(doc_id, "|", sentence, "| Found:", found)


Strategy: 0
assignments | However, students with verified medical emergencies may request an extension | Found: True

Strategy: 10
assignments | However, students with verified medical emergencies may request an extension | Found: True

Strategy: sentence
assignments | However, students with verified medical emergencies may request an extension | Found: True


In [58]:
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(docs, strategy)

    print("\nStrategy:", strategy)

    for c in chunks:
        if c["doc"] == "assignments":
            if "Late submissions normally receive a penalty" in c["text"]:
                print("Rule and exception together:",
                      "However, students with verified medical emergencies may request an extension"
                      in c["text"])


Strategy: 0
Rule and exception together: True

Strategy: 10
Rule and exception together: True

Strategy: sentence
Rule and exception together: True


In [59]:


print("Empty text:", windows("") == [])

print("Short text:", windows("hello world") == [(0, 11)])

try:
    windows("hello world", 40, 40)
except ValueError:
    print("Invalid overlap: Passed")
else:
    print("Invalid overlap: Failed")

long_sentence = " ".join(["word"] * 95) + "."
long_chunks = sentence_chunks(long_sentence)

print("Long sentence: Passed" if all(
    len(c[0:0]) == 0 for c in []
) and all(
    len(long_sentence[a:b].split()) <= 40
    for a, b in long_chunks
) else "Long sentence: Failed")

Empty text: True
Short text: True
Invalid overlap: Passed
Long sentence: Passed


In [78]:
%%writefile guided_embeddings.py
mport numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
passages = ["Reset a forgotten password using the recovery link.",
 "Submit assignments through the learning portal.",
 "Check ink levels when a printer produces blank pages."]
C = model.encode(passages, normalize_embeddings=True)
Q = model.encode(["How can I recover my login?"],
                 normalize_embeddings=True)
scores = (Q @ C.T)[0]
print("Embedding shape:", C.shape)
for i in np.argsort(-scores):
    print(i, round(float(scores[i]), 3), passages[i])
assert np.allclose(np.linalg.norm(C, axis=1), 1, atol=1e-5)


Writing guided_embeddings.py


In [62]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")
    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
 ["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")
    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i,r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key,value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]
def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])


Overwriting lab6_search.py


In [63]:
%%writefile guided_chatbot.py
from lab6_search import Retriever
FAQ = [
{"id":"faq:password", "question":"How do I reset my password?",
 "answer":"Use the recovery link on the login page.",
 "source":"it_help.txt"},
{"id":"faq:submit", "question":"Where do I submit my assignment?",
 "answer":"Use the assignment upload page in the learning portal.",
 "source":"assessment.txt"},
{"id":"faq:late", "question":"Can illness justify a late submission?",
 "answer":"Request an extension with medical evidence before the deadline.",
 "source":"handbook.txt"}]
def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"]+" "+r["answer"]) for r in FAQ]
    return Retriever(records, backend)
def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r,score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf","embedding"],
                             default="tfidf")
    args = parser.parse_args()
    engine = build_bot(args.backend)
    print("FAQ assistant. Type exit to stop.")
    while True:
        try:
            query = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "exit":
            break
        print("Bot:", reply(engine, query))

Overwriting guided_chatbot.py


In [64]:
%%writefile guided_suggestions.py
from guided_chatbot import build_bot
engine = build_bot("embedding")
query = "assignment upload"
print("Suggested questions:")
for record, score in engine.search(query, k=3):
    if score > 0: # lexical overlap only; not a semantic cutoff
        print(record["question"], f"({score:.3f})")

Overwriting guided_suggestions.py


In [65]:
!python guided_suggestions.py

Loading weights: 100% 103/103 [00:00<00:00, 8035.81it/s]
Suggested questions:
Where do I submit my assignment? (0.671)
Can illness justify a late submission? (0.086)


In [69]:
%%writefile task2_semantic_check.py

from task2_chunks import make_chunks
from lab6_search import Retriever, show_hits
def compare_retrieval(docs, queries):
    records = make_chunks(docs, "sentence")
    for backend in ["tfidf", "embedding"]:
        engine = Retriever(records, backend)
        print("\nBackend:", backend)
        for query in queries:
            print("\nQuery:", query)
            hits = engine.search(query, k=3)
            show_hits(hits)

Overwriting task2_semantic_check.py


In [74]:
from task2_semantic_check import compare_retrieval

queries = [
    "What happens if a student cannot meet the attendance requirement?",
    "Can I get an extension if I have a medical emergency?",
    "What rules should students follow when using the library?"
]

compare_retrieval(docs, queries)


Backend: tfidf

Query: What happens if a student cannot meet the attendance requirement?
attendance:3 score=0.442
Students should inform their instructor if they cannot attend a class.
    Attendance rules apply to all regular classes and laboratory sessions.
    A student should maintain good attendance throughout the semester.
    The department may provide guidance to students with low attendance.
    
attendance:1 score=0.281
Students should check their attendance regularly.
    If a student has a genuine medical emergency, the department may review the case.
    Medical documents must be submitted for verification.
    The department will make the final decision after reviewing the evidence.
    
attendance:2 score=0.235
Students should contact their class coordinator for attendance-related questions.
    Students can check their attendance through the university portal.
    Any mistake in the attendance record should be reported immediately.
    The instructor may correct errors

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Backend: embedding

Query: What happens if a student cannot meet the attendance requirement?
attendance:3 score=0.716
Students should inform their instructor if they cannot attend a class.
    Attendance rules apply to all regular classes and laboratory sessions.
    A student should maintain good attendance throughout the semester.
    The department may provide guidance to students with low attendance.
    
attendance:0 score=0.681

    Students are expected to attend all scheduled classes.
    A minimum attendance of 75 percent is required to appear in final examinations.
    Students who arrive late may be marked absent by the instructor.
    Attendance records are maintained by the department.
    
attendance:2 score=0.601
Students should contact their class coordinator for attendance-related questions.
    Students can check their attendance through the university portal.
    Any mistake in the attendance record should be reported immediately.
    The instructor may correct erro

In [75]:
from task2_chunks import make_chunks

chunks = make_chunks(docs, "sentence")

for c in chunks:
    original = docs[c["doc"]][c["start"]:c["end"]]

    assert original == c["text"]

    print(
        c["id"],
        "Start:", c["start"],
        "End:", c["end"],
        "Evidence:", c["text"][:100]
    )

attendance:0 Start: 0 End: 274 Evidence: 
    Students are expected to attend all scheduled classes.
    A minimum attendance of 75 percent i
attendance:1 Start: 274 End: 550 Evidence: Students should check their attendance regularly.
    If a student has a genuine medical emergency, 
attendance:2 Start: 550 End: 851 Evidence: Students should contact their class coordinator for attendance-related questions.
    Students can c
attendance:3 Start: 851 End: 1145 Evidence: Students should inform their instructor if they cannot attend a class.
    Attendance rules apply to
assignments:0 Start: 0 End: 328 Evidence: 
    Students must submit assignments before the announced deadline.
    Late submissions normally r
assignments:1 Start: 328 End: 646 Evidence: Students should upload their assignments through the learning portal.
    The portal provides a subm
assignments:2 Start: 646 End: 901 Evidence: Every assignment must include the student's name and roll number.
    Work should be complete

In [77]:
text = " ".join(["word"] * 38) + ". Dr. Ahmed teaches AI."

chunks = sentence_chunks(text)

for a, b in chunks:
    print(text[a:b])

word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word. Dr. 
Ahmed teaches AI.


In [79]:
!ls


guided_chatbot.py      lab6_search.py  task2_chunks.py
guided_embeddings.py   __pycache__     task2_semantic_check.py
guided_suggestions.py  sample_data
